In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import os
import glob
import shutil

In [ ]:
# O modelo base que você usou no treinamento (Gemma 3.4B-IT)
BASE_MODEL = "google/gemma-3-4b-it" 
# Onde você salvou o adaptador LoRA
LORA_PATH = "/mnt/data/jupyter-env/donaldDuck/donald_lora_out"
# Onde o modelo final fundido será salvo (novo diretório!)
MERGED_MODEL_PATH = "/mnt/data/jupyter-env/donaldDuck/donald_duck_merged_model" 

# 1. Carregar o modelo base e o tokenizer
print(f"Carregando modelo base: {BASE_MODEL}")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16, # Use o dtype de treinamento
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(LORA_PATH) # O tokenizer foi salvo junto com o LoRA

# 2. Carregar o adaptador LoRA
print(f"Carregando adaptador LoRA de: {LORA_PATH}")
model = PeftModel.from_pretrained(base_model, LORA_PATH)

In [ ]:
# 3. Fundir (Merge) os pesos
# Esta é a operação que aplica os pesos LoRA no modelo base.
print("Fundindo (Merging) pesos LoRA no modelo base...")
model = model.merge_and_unload()

os.makedirs(MERGED_MODEL_PATH, exist_ok=True)

# 4. Salvar o modelo fundido completo
print(f"Salvando modelo fundido em: {MERGED_MODEL_PATH}")
model.save_pretrained(MERGED_MODEL_PATH, safe_serialization=True)
tokenizer.save_pretrained(MERGED_MODEL_PATH)

In [ ]:
print("Garantindo a presença do 'tokenizer.model' para conversão GGUF...")

# Caminho para o cache do Hugging Face (padrão)
HF_CACHE_DIR = os.path.expanduser("~/.cache/huggingface/hub/")

# A maneira mais robusta de encontrar o arquivo:
# Procurar pelo diretório de cache do modelo base e então encontrar o snapshot
try:
    # 1. Tenta encontrar a pasta do repositório Gemma
    gemma_repo_path = glob.glob(os.path.join(HF_CACHE_DIR, f"models--{BASE_MODEL.replace('/', '--')}/snapshots/*"))
    
    if gemma_repo_path:
        # Pega a pasta mais recente (o último snapshot)
        snapshot_path = gemma_repo_path[-1] 
        source_file = os.path.join(snapshot_path, "tokenizer.model")
        
        destination_file = os.path.join(MERGED_MODEL_PATH, "tokenizer.model")

        if os.path.exists(source_file):
            shutil.copy(source_file, destination_file)
            print(f"✅ '{MERGED_MODEL_PATH}/tokenizer.model' copiado com sucesso para o diretório final.")
        else:
            print(f"❌ Erro: Não foi possível encontrar 'tokenizer.model' no cache: {source_file}")
            print("A conversão GGUF pode falhar!")

except Exception as e:
    print(f"Erro inesperado durante a busca do tokenizer.model: {e}")
    print("A conversão GGUF pode falhar, tente a cópia manual.")

In [ ]:
print("Fusão concluída. O modelo completo está pronto para a quantização GGUF!")